In [1]:
import pandas as pd
import numpy as np

In [2]:
#this index is just a tuple. A sequence of objects
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]

populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)

index = pd.MultiIndex.from_tuples(index)
pop = pop.reindex(index)
pop
#the first two columns show the multiple index value.The third column shows the data

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [3]:
#to access every row where the 2nd index = 2010
pop[:,2010]  

California    37253956
New York      19378102
Texas         25145561
dtype: int64

In [4]:
#can easily convert a multi-index into a normal pandas dataframe
#converts each 2nd layer into its own column
pop_df = pop.unstack()
pop_df

,2000,2010
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


In [5]:
#you can undo this with .stack()
pop_df.stack()

California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [6]:
#multiindexes are useful b/c they allow us to add in additional dimensions into our data
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9267089, 9284094,
                                   4687374, 4318033,
                                   5906301, 6879014]})
pop_df

total  under18
California 2000  33871648  9267089
           2010  37253956  9284094
New York   2000  18976457  4687374
           2010  19378102  4318033
Texas      2000  20851820  5906301
           2010  25145561  6879014

In [7]:
#ufuncs also work with multi-index data
f_u18 = pop_df['under18']/pop_df['total']
f_u18.unstack()

,2000,2010
California,0.273594,0.249211
New York,0.247010,0.222831
Texas,0.283251,0.273568


In [8]:
#can create a multi-index by passing multiple arrays into the index during DF creation
df = pd.DataFrame(np.random.rand(4, 2),
                 index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]], #passing multiple arrays into the index
                 columns = ['data1','data2'])
df

data1     data2
a 1  0.511865  0.971885
  2  0.889119  0.781634
b 1  0.361345  0.467329
  2  0.804744  0.235136

In [9]:
#you can also create a multi-index by passing in a dictionary where you have multi-level keys
data = {('California', 2000): 33871648,
        ('California', 2010): 37253956,
        ('Texas', 2000): 20851820,
        ('Texas', 2010): 25145561,
        ('New York', 2000): 18976457,
        ('New York', 2010): 19378102}
pd.Series(data)

California  2000    33871648
            2010    37253956
Texas       2000    20851820
            2010    25145561
New York    2000    18976457
            2010    19378102
dtype: int64

In [10]:
#you can also explicitly create a multi-index
#this is if you are passing in two arrays
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [11]:
#This is if you decide to use a tuple
pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [12]:
#you can create a multi-index using the cartesian product of multiple arrays
pd.MultiIndex.from_product([['a', 'b'], [1, 2,3]])

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 2),
            ('b', 3)],
           )

In [13]:
#you can also directly create a multi-index by directly setting it's levels and labels
pd.MultiIndex(levels=[['a', 'b'], [1, 2]],
              codes=[[0, 0, 1, 1], [0, 1, 0, 1]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

In [14]:
#when creating a multi-index, you can also set names if so fit
pop.index.names = ['state', 'year']
pop

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
Texas       2000    20851820
            2010    25145561
dtype: int64

In [15]:
#columns can also have multi-indices
# hierarchical indices and columns
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# mock some data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

# create the DataFrame
health_data = pd.DataFrame(data, index=index, columns=columns) #this is where we pass in the multi-index
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      49.0  36.6  54.0  36.1  45.0  36.6
     2      51.0  36.5  40.0  35.3  38.0  37.9
2014 1      30.0  36.0  27.0  36.8  25.0  37.2
     2      36.0  36.0  47.0  38.8  29.0  35.4

In [16]:
#it is possible to filter to only 'Guido'
health_data['Guido']

type          HR  Temp
year visit            
2013 1      54.0  36.1
     2      40.0  35.3
2014 1      27.0  36.8
     2      47.0  38.8

In [17]:
pop
#to slice to a single element
pop['California',2000]

#to slice to multiple rows
pop['California']

#to slice to multiple rows pertaining to multiple indices
pop.loc['California':'New York']

#you can also slice on lower level indices
pop.loc[:,2000]

#you can also slice w/ a boolean mask
pop.loc[pop>30000000]

#can also slice with just the index names
pop[['California','New York']]

state       year
California  2000    33871648
            2010    37253956
New York    2000    18976457
            2010    19378102
dtype: int64

In [18]:
#slicing behaviour with multiple indices
health_data
health_data['Bob','HR']

year  visit
2013  1        49.0
      2        51.0
2014  1        30.0
      2        36.0
Name: (Bob, HR), dtype: float64

In [19]:
health_data
#gives you the first 2 rows and the first 2 columns ~ eg: row index 0,1 and column index 0,1
health_data.iloc[:2,:2]

subject      Bob      
type          HR  Temp
year visit            
2013 1      49.0  36.6
     2      51.0  36.5

In [20]:
#returns every row, and the columns associated with Bob/HR
health_data.loc[:, ('Bob', 'HR')]

year  visit
2013  1        49.0
      2        51.0
2014  1        30.0
      2        36.0
Name: (Bob, HR), dtype: float64

In [30]:
# use index slice to work with getting out data that is within a tuple
idx = pd.IndexSlice
#returns the first row of each Year tuple
#returns the first row of each type tuple
health_data.loc[idx[:, 1], idx[:, 'HR']]

,subject,Bob,Guido,Sue
,type,HR,HR,HR
year,visit,,,
2013,1,49.0,54.0,45.0
2014,1,30.0,27.0,25.0


In [33]:
#the rearrangement of a muti-index series will fail if the index is not sorted
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']
data

try:
    data['a':'b']
except KeyError as e:
    print(type(e))
    print(e)

<class 'pandas.errors.UnsortedIndexError'>
'Key length (1) was greater than MultiIndex lexsort depth (0)'


In [36]:
#to sort an index you can use sort_index()
data = data.sort_index()
data['a':'b']

char  int
a     1      0.110893
      2      0.785394
b     1      0.250592
      2      0.444498
dtype: float64

In [41]:
#to convert a multi-index into a simpler 2-d table use unstack
#you can use the level parameter to determine which index to use
pop.unstack(level=1)

year,2000,2010
state,,
California,33871648,37253956
New York,18976457,19378102
Texas,20851820,25145561


In [43]:
#use stack to convert a simple 2d representation back to the multi-index
pop.unstack(level=0).stack()

year  state     
2000  California    33871648
      New York      18976457
      Texas         20851820
2010  California    37253956
      New York      19378102
      Texas         25145561
dtype: int64

In [49]:
#another way to convert a multi-index into a 2d dataframe is to convert the multi-index indices into columns
pop_flat = pop.reset_index(name='population')
pop_flat

,state,year,population
0,California,2000,33871648
1,California,2010,37253956
2,New York,2000,18976457
3,New York,2010,19378102
4,Texas,2000,20851820
5,Texas,2010,25145561


In [52]:
#you can convert a flat 2d dataset to use the multi-index via set_index
pop_flat.set_index(['state','year'])

population
state      year            
California 2000    33871648
           2010    37253956
New York   2000    18976457
           2010    19378102
Texas      2000    20851820
           2010    25145561

In [54]:
#with multi-index datasets you can apply aggregate functions only for certain index levels
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      49.0  36.6  54.0  36.1  45.0  36.6
     2      51.0  36.5  40.0  35.3  38.0  37.9
2014 1      30.0  36.0  27.0  36.8  25.0  37.2
     2      36.0  36.0  47.0  38.8  29.0  35.4

In [57]:
#mean accross years
data_mean = health_data.mean(level = 'year')

#you can go one step further and take the mean across columsn
data_mean.mean(axis=1, level = 'type')

#note that the above functions are the equivalent to using pandas group by

type,HR,Temp
year,,
2013,46.166667,36.5
2014,32.333333,36.7
